# Phase 2: resource limits and shrimp mortality

This notebook extends the base shrimp/algae/bird model with logistic algae growth and natural shrimp mortality.

Can adding a algae carrying capacity and natural shrimp death rate sustain stable coexistence across a range of bird mortality rates?


## Units and parameter guesses

Time is in **days**; algae $A$ is water chlorophyll-a in **$\mu$g Chl-a/L**; shrimp $S$ is **individuals/L**; birds $B$ is **individuals**. Chlorophyll represents the algae community, with *Dunaliella viridis* as the focal food species. All values below are provisional, not calibrated.

| Parameter | Meaning | Value | Units | Brief justification |
|---|---|---|---|---|
| `beta_A` | Algae growth rate | 0.3 | day$^{-1}$ | Guess: about 2.3-day doubling at low abundance |
| `K` | Algae carrying capacity | 50 | $\mu$g Chl-a/L | Exploratory guess; observed peaks do not directly measure capacity |
| `C_A` | Shrimp clearance coefficient | 0.02 | L/(shrimp day) | Intermediate guess informed by laboratory clearance scales (Makridis & Vadstein, 1999) |
| `C_S` | Effective bird feeding coefficient | 0.4 | 1/(bird day) | Earlier illustrative value; needs feeding and habitat-volume estimates |
| `gamma_S` | Shrimp gain per chlorophyll loss | 0.2 | shrimp/($\mu$g Chl-a) | Earlier illustrative conversion; needs production data |
| `gamma_B` | Bird gain per shrimp density loss | 0.3 | bird L/shrimp | Earlier illustrative conversion; needs demographic and volume estimates |
| `delta_S` | Shrimp natural mortality | 0.03 | day$^{-1}$ | Exploratory guess; separate from predation |
| `delta_B` | Bird natural mortality | 0.12 | day$^{-1}$ | Earlier illustrative value; separate from migration |

Initial guesses: $S_0=1$ shrimp/L, $A_0=10$ $\mu$g Chl-a/L, $B_0=1$ bird. Salinity sources report experimental values in **ppt**; salinity is not yet included in the equations. Full sources, study details, exploratory ranges, and conversion notes appear at the bottom.


## Equations

The state order throughout the code is **shrimp, algae, birds**: $(S,A,B)$.

$$
\frac{dS}{dt} = \gamma_S C_A S A - C_S S B - \delta_S S
$$

$$
\frac{dA}{dt} = \beta_A A(1-A/K) - C_A S A
$$

$$
\frac{dB}{dt} = -\delta_B B + \gamma_B C_S S B
$$

$K>0$ is the algae carrying capacity in the absence of grazing. $\delta_S>0$ is the shrimp natural mortality rate. Feeding and conversion terms retain their phase 1 meanings. These changes do not yet include salinity, lake volume, seasonal forcing, or harvesting.

In [ ]:
from dataclasses import dataclass, replace
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

@dataclass(frozen=True)
class Parameters:
    beta_A: float = 0.3  # Guessed daily algae growth
    C_A: float = 0.02  # Provisional clearance guess, L/(shrimp day)
    C_S: float = 0.4
    gamma_S: float = 0.2  # Uncalibrated chlorophyll-to-shrimp conversion
    gamma_B: float = 0.3
    delta_B: float = 0.12
    K: float = 50.0       # New: illustrative algae carrying capacity
    delta_S: float = 0.03  # New: illustrative shrimp mortality

p = Parameters()
y0 = np.array([1.0, 10.0, 1.0])  # Guesses: shrimp/L, micrograms Chl-a/L, birds
T = 400.0
print(p)

In [ ]:
def rhs(t, state, p):
    S, A, B = state
    return np.array([
        p.gamma_S*p.C_A*S*A - p.C_S*S*B - p.delta_S*S,
        p.beta_A*A*(1 - A/p.K) - p.C_A*S*A,
        -p.delta_B*B + p.gamma_B*p.C_S*S*B,
    ])

def simulate(p, initial=y0, duration=T):
    if any(v <= 0 for v in vars(p).values()):
        raise ValueError('All parameters must be positive.')
    initial = np.asarray(initial, dtype=float)
    if initial.shape != (3,) or np.any(initial < 0):
        raise ValueError('Initial state must contain three nonnegative populations.')
    sol = solve_ivp(lambda t, y: rhs(t, y, p), (0, duration), initial,
                    t_eval=np.linspace(0, duration, 2401), rtol=1e-8, atol=1e-10)
    if not sol.success:
        raise RuntimeError(sol.message)
    if np.min(sol.y) < -1e-7:
        raise RuntimeError('Negative numerical populations: inspect solver settings.')
    return sol

sol = simulate(p)

## Coexistence equilibrium

For all three populations to be positive, dividing the equilibrium equations by their respective populations gives:

$$
S^* = \frac{\delta_B}{\gamma_B C_S}
$$

$$
A^* = K\left(1-\frac{C_A S^*}{\beta_A}\right)
$$

$$
B^* = \frac{\gamma_S C_A A^* - \delta_S}{C_S}
$$

Thus positive coexistence requires

$$
\gamma_S C_A K > \delta_S
$$

and

$$
0 < \delta_B < \frac{\gamma_B C_S \beta_A}{C_A}
\left(1-\frac{\delta_S}{\gamma_S C_A K}\right).
$$

This replaces the phase 1 equality condition with an interval. At any strictly positive coexistence equilibrium, the Jacobian characteristic polynomial has the form $\lambda^3+x\lambda^2+(uw+vy)\lambda+xvy$, with $x=\beta_A A^*/K$, $u=\gamma_S C_A S^*$, $v=C_S S^*$, $w=C_A A^*$, and $y=\gamma_B C_S B^*$. All are positive and $x(uw+vy)>xvy$, so the Routh?Hurwitz criterion gives **local asymptotic stability**. This is a local result; trajectories below provide numerical examples, not a proof of global convergence.

In [ ]:
def coexistence(p):
    S = p.delta_B/(p.gamma_B*p.C_S)
    A = p.K*(1 - p.C_A*S/p.beta_A)
    B = (p.gamma_S*p.C_A*A - p.delta_S)/p.C_S
    return np.array([S, A, B])

def jacobian(state, p):
    S, A, B = state
    return np.array([
        [p.gamma_S*p.C_A*A-p.C_S*B-p.delta_S, p.gamma_S*p.C_A*S, -p.C_S*S],
        [-p.C_A*A, p.beta_A*(1-2*A/p.K)-p.C_A*S, 0],
        [p.gamma_B*p.C_S*B, 0, -p.delta_B+p.gamma_B*p.C_S*S],
    ])

eq = coexistence(p)
threshold = p.gamma_B*p.C_S*p.beta_A/p.C_A*(1-p.delta_S/(p.gamma_S*p.C_A*p.K))
print('Coexistence candidate [S, A, B]:', eq)
print('Upper bird mortality bound:', threshold)
if np.all(eq > 0):
    assert np.allclose(rhs(0, eq, p), 0, atol=1e-10)
    eigenvalues = np.linalg.eigvals(jacobian(eq, p))
    print('Jacobian eigenvalues:', eigenvalues)
    print('Locally stable:', np.all(eigenvalues.real < 0))
else:
    print('The coexistence candidate is not strictly positive for these parameters.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for i, label in enumerate(['Shrimp', 'Algae', 'Birds']):
    line, = ax.plot(sol.t, sol.y[i], label=label)
    if np.all(eq > 0):
        ax.axhline(eq[i], color=line.get_color(), linestyle=':', alpha=0.5)
ax.set(xlabel='Time (days)', ylabel='State value (units differ; see parameter table)',
       title='Phase 2: time series; dotted lines show coexistence equilibrium')
ax.legend()
fig.tight_layout()
plt.show()

## Shrimp/algae phase portrait without birds

Setting $B=0$ gives an invariant two-dimensional subsystem. Its nonzero nullclines are $A=\delta_S/(\gamma_S C_A)$ and $S=(\beta_A/C_A)(1-A/K)$. A positive shrimp?algae equilibrium exists when $\gamma_S C_A K>\delta_S$. This plot concerns the bird-free subsystem; it is not a slice through the full three-species dynamics.

In [ ]:
Smax = 1.5*p.beta_A/p.C_A
Amax = 1.15*p.K
sg = np.linspace(0, Smax, 25)
ag = np.linspace(0, Amax, 25)
SS, AA = np.meshgrid(sg, ag)
dS = SS*(p.gamma_S*p.C_A*AA-p.delta_S)
dA = AA*(p.beta_A*(1-AA/p.K)-p.C_A*SS)
# Normalize by each axis range to keep direction arrows readable.
speed = np.hypot(dS/Smax, dA/Amax)
scale = np.maximum(speed, 1e-12)
fig, ax = plt.subplots(figsize=(7, 5))
ax.quiver(SS, AA, dS/scale, dA/scale, alpha=0.35, angles='xy')
ax.axhline(p.delta_S/(p.gamma_S*p.C_A), color='tab:orange', linestyle='--', label="S'=0 (nonzero)")
a_line = np.linspace(0, p.K, 200)
ax.plot(p.beta_A/p.C_A*(1-a_line/p.K), a_line, '--', color='tab:green', label="A'=0 (nonzero)")
for initial in ([0.3, 2, 0], [1.0, 8, 0], [2.5, 5, 0]):
    trajectory = simulate(p, initial)
    ax.plot(trajectory.y[0], trajectory.y[1], linewidth=1.5)
ax.set(xlim=(0, Smax), ylim=(0, Amax), xlabel='Shrimp S', ylabel='Algae A',
       title='Bird-free subsystem: trajectories and nullclines')
ax.legend()
fig.tight_layout()
plt.show()

## Change bird mortality

Compare mortality rates below and above the positive coexistence threshold. The simulations use the same initial populations. Finite-time small populations should not be interpreted as an exact extinction event.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharex=True, sharey=True)
for ax, mortality in zip(axes, [0.5*threshold, 0.9*threshold, 1.1*threshold]):
    trial = replace(p, delta_B=mortality)
    result = simulate(trial)
    for i, label in enumerate(['Shrimp', 'Algae', 'Birds']):
        ax.plot(result.t, result.y[i], label=label)
    ax.set(title=f'Bird mortality = {mortality}', xlabel='Time (days)')
axes[0].set_ylabel('State value (units differ; see parameter table)')
axes[-1].legend()
fig.tight_layout()
plt.show()

## Next experiments

- Change `K` and `delta_S` in the parameter cell, then rerun all cells.
- Compare several positive initial conditions at fixed parameters.
- Investigate the boundary equilibria when positive coexistence fails.
- Use water chlorophyll-a and live-shrimp density observations, define the bird habitat volume, and estimate daily parameters before making ecological predictions.

Plots are embedded in the notebook; this notebook does not overwrite or export project images.

## Sources and calibration notes

For this exploration, the algae state $A$ is **water-column chlorophyll-a concentration**, used directly as a proxy for algae abundance. Its units are **micrograms of chlorophyll-a per liter** ($\mu$g Chl-a/L). It is not total algal dry mass or carbon. Water chlorophyll includes multiple phytoplankton species; treating it as the food available to shrimp is a simplifying assumption.

*Dunaliella viridis* remains the focal algae species motivating the model. [Utah DWR, Algae and algal blooms](https://wildlife.utah.gov/gslep/wildlife/algae) identifies it as a preferred food of brine shrimp. The chlorophyll proxy is not a species-specific measurement of *D. viridis*.

### Sources for chlorophyll, salinity, and shrimp observations

- [Belovsky et al. (2025), Environment and phytoplankton relative abundances in a hypersaline lake: 27 years in Great Salt Lake, USA and experiments](https://link.springer.com/article/10.1007/s10452-025-10190-1): a source for chlorophyll-a measurements, salinity information, and experiments with *D. viridis*. Laboratory chlorophyll is measured in micrograms Chl-a/L. Experimental salinities are 30, 60, 90, 120, and 150 **ppt** (parts per thousand); these are experimental treatments, not a chosen present-day lake salinity. The study also analyzes South Arm field monitoring from 1994-2020. Its experimental absolute growth measure is not directly the per-capita `beta_A` in our equations.
- [Utah DWQ Great Salt Lake Data Explorer](https://udwq.shinyapps.io/GSL_data_explorer/): a place to look for lake chlorophyll-a and salinity observations. [DWQ monitoring documentation](https://lf-public.deq.utah.gov/WebLink/ElectronicFile.aspx?dbid=0&docid=14163&eqdocs=DWQ-2021-018149&repo=Public) describes these parameters. Check the downloaded record's units, date, location, depth, and method; the dashboard was not directly inspected. Do not interchange ppt and g/L without checking the reported salinity definition and brine density.
- [Utah DWR brine shrimp sampling reports](https://wildlife.utah.gov/gslep/harvest): observations **per liter of lake water**, separated by life stage and sex. Use the appropriate live-shrimp categories for $S$ and avoid double-counting female subcategories. Cysts are separate from live shrimp.

### Unit conversions and model assumptions

`gamma_S` and `gamma_B` are unit-dependent conversion factors, not necessarily dimensionless efficiencies or bounded by one. Chlorophyll-based conversions are effective model relationships rather than direct food-mass efficiencies.

Because birds are counts and shrimp is density, the bird predation terms still require a defined habitat volume when translating physical consumption into model coefficients. For a fixed volume $V$ in liters, total bird consumption $Q$ shrimp/day removes density at $Q/V$ shrimp/L/day. Thus `C_S`, `gamma_B`, and `h_S` include spatial conversions. Retaining the current equations is acceptable for exploration with effective coefficients, but a changing-volume model would require an explicit mass balance.

Next steps: select compatible lake observations, document their region, season, depths, and units, then estimate daily parameter ranges with sources and uncertainty. Salinity is recorded as context for now; the current equations do not yet depend on it.

### First guesses and feeding experiments

These are starting guesses for exploration, not fitted estimates or confidence intervals. Change one at a time before attempting a fit to observations.

| Parameter | First guess | Suggested exploratory range | Basis |
|---|---|---|---|
| `C_A` | 0.02 L/(shrimp day) | 0.002-0.05 | Order-of-magnitude guess informed by clearance experiments; not a measured *D. viridis* lake-average rate |
| `beta_A` | 0.3/day | 0.1-0.6/day | Unverified guess; low-density algae doubling time about 2.3 days |
| `K` | 50 micrograms Chl-a/L | 20-100 micrograms Chl-a/L | Unverified carrying-capacity guess; field chlorophyll peaks are not ungrazed carrying capacity |
| `delta_S` | 0.03/day | 0.01-0.1/day | Unverified natural-mortality guess; excludes bird predation |

Other coefficients, including conversion factors, bird feeding/mortality, and Phase 3 handling times, retain the earlier illustrative values. They have no experimental calibration. Initial conditions are also guesses: 1 shrimp/L, 10 micrograms Chl-a/L, and 1 bird in an unspecified habitat. Bird-related results cannot be interpreted quantitatively until habitat volume and bird parameters are specified. No values were selected to guarantee coexistence.

**Experiments located:**

1. [Sura, Herlihy, Mahon and Belovsky (2017), Environmental impacts on grazing of different brine shrimp (Artemia franciscana) life stages](https://link.springer.com/article/10.1007/s10750-016-3047-5). Directly relevant: Great Salt Lake shrimp feeding on *D. viridis* and *Coccochloris elabens*, with life stage, salinity, temperature and food concentration varied. One-hour trials show that one universal feeding coefficient is a simplification. The publisher provides an abstract/preview; no numerical ingestion rate was extracted from its full text here. Best next source for condition-specific feeding estimates, potentially through university access.

2. [Belovsky, Stumpf and Girgis (published online 2024; volume 2025), Artemia selective grazing: survival value and nutritional intake](https://link.springer.com/article/10.1007/s10750-024-05719-8). Open-access experiments with *D. viridis*, cyanobacteria and diatoms, singly and mixed, across shrimp developmental stages. Reports ingestion as biovolume/individual/hour and describes chlorophyll, cell-count and biovolume conversions. Particularly useful for our community-chlorophyll approximation; rates must be converted before insertion into the equations.

3. [Makridis and Vadstein (1999), Food size selectivity of Artemia franciscana at three developmental stages](https://academic.oup.com/plankt/article-abstract/21/11/2191/1499662). The abstract reports maximum clearance rates of 50-63 microliters/individual/hour at 2 days old, 254 at 4 days, and 1.48-2.10 milliliters/individual/hour at 7 days. Converted by multiplying by 24 and converting to liters, these are 0.0012-0.001512, 0.006096, and 0.03552-0.0504 L/(individual day). These are laboratory maximum rates across tested particles, not a *D. viridis*-specific field estimate. They support exploring a broad feeding scale; `C_A=0.02` is our intermediate guess, not a reported experimental result.

For Phase 2, ingestion $I$ in micrograms Chl-a/(shrimp day) at food concentration $A$ gives $C_A=I/A$; clearance in L/(shrimp day) maps directly to `C_A` under the proportional-feeding assumption. For Phase 3, fit $I(A)=C_A A/(1+h_A C_A A)$ to multiple concentrations: `C_A` is the low-food slope and $1/h_A$ is the maximum ingestion rate. A single clearance measurement does not identify both parameters. For example, `C_A=0.02` predicts 0.2 micrograms Chl-a/(shrimp day) at $A=10$ in Phase 2; Phase 3's current guessed handling time reduces that to about 0.182.
